# Similarity-aware Label Smoothing

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
import numpy as np
from tqdm import tqdm
from dataset_utils import get_data_loaders
from hyperparams import dataset, batch_size, device

## Hyperparams

In [ ]:
lr = 5e-4
epochs = 20


## Models

In [4]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2,2)
        self.fc1   = nn.Linear(64*7*7, 128)
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)


### CIFAR-100

In [5]:
def RESNET18():
    model = models.resnet18(weights = None)

    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, 100)

    return model

## Training Utils

### Accuracy Measure

In [8]:
def accuracy(model, loader, k = (1, 5)):
    model.eval()
    correct = {key:0 for key in k}
    total = 0

    maxk = max(k)

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)

            _, pred = outputs.topk(maxk, dim=1, largest=True, sorted=True)

            y_expand = y.view(-1, 1).expand_as(pred)

            for key in k:
                correct[key] += (pred[:, :key] == y.view(-1, 1)).any(dim=1).sum().item()
            total += y.size(0)

    return {key: correct[key] / total for key in k}

### Label Smoothing

In [9]:
def uniform_smooth_labels(y, num_classes = 10, epsilon = 0.1):
    y_onehot = F.one_hot(y, num_classes).float()
    return ((num_classes * (1 - epsilon) - 1) * y_onehot + epsilon) / (num_classes - 1)

## Training Loop

In [10]:
def train(model, train_loader, test_loader, optimizer=None, scheduler=None, num_classes=100, epochs=10, epsilon=0.1):
    for epoch in range(epochs):
        model.train()
        running_loss = 0

        for x, y in tqdm(train_loader, leave=False):
            x, y = x.to(device), y.to(device)

            targets = uniform_smooth_labels(y, num_classes=num_classes, epsilon=epsilon)

            optimizer.zero_grad()
            logits = model(x)
            loss = -(targets * F.log_softmax(logits, dim=1)).sum(dim=1).mean()
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * x.size(0)

        if scheduler is not None: scheduler.step()

        test_acc = accuracy(model, test_loader)
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {running_loss/len(train_loader.dataset):.4f} | Test Acc: {test_acc[1]:.4f} | Top-5 Test Acc: {test_acc[5]:.4f}")

## RUN Experiments

In [ ]:
train_loader, test_loader = get_data_loaders(dataset=dataset)

model = RESNET18().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.2, momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

train(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    num_classes=100,
    epochs=10,
    epsilon=0,
)

Epoch 1/10 | Loss: 4.1096 | Test Acc: 0.1169 | Top-5 Test Acc: 0.3472


Epoch 2/10 | Loss: 3.4934 | Test Acc: 0.2043 | Top-5 Test Acc: 0.4889


Epoch 3/10 | Loss: 2.9675 | Test Acc: 0.2814 | Top-5 Test Acc: 0.5993


Epoch 4/10 | Loss: 2.4678 | Test Acc: 0.3183 | Top-5 Test Acc: 0.6385


Epoch 5/10 | Loss: 2.1096 | Test Acc: 0.4326 | Top-5 Test Acc: 0.7535


Epoch 6/10 | Loss: 1.8418 | Test Acc: 0.4675 | Top-5 Test Acc: 0.7880


Epoch 7/10 | Loss: 1.5963 | Test Acc: 0.5240 | Top-5 Test Acc: 0.8257


Epoch 8/10 | Loss: 1.3442 | Test Acc: 0.5598 | Top-5 Test Acc: 0.8477


Epoch 9/10 | Loss: 1.1094 | Test Acc: 0.6244 | Top-5 Test Acc: 0.8864


Epoch 10/10 | Loss: 0.9264 | Test Acc: 0.6650 | Top-5 Test Acc: 0.9014
